# **Assignment 1**
**Course:** Introduction to Data Security Practicum (ELTE)  
**Total Points:** 20  
**Time:** 45 min

---

1. **Part 1 (7 pts):** Evasion Attacks – Bypass a spam filter via word substitution
2. **Part 2 (5 pts):** Data Poisoning – Corrupt training data to degrade a model
3. **Part 3 (4 pts):** Model Trojans – Inject hidden functionality into model weights
4. **Part 4 (4 pts):** Integration & Defense – Design a defense strategy

Each part includes scaffolded code with `TODO` comments. Follow the instructions and fill in the blanks.

## **PART 1: Evasion Attacks (7 pts)**

Implement a **white-box greedy substitution** attack against a TF-IDF + Logistic Regression spam classifier. Replace "spammy" words with "hammy" words until the filter is fooled.

- Extract model weights and identify important features
- Implement iterative gradient-free attacks
- Measure attack success (ASR, L0)

In [68]:
import pandas as pd
import numpy as np
import joblib
import re

# Load the provided pre-trained model and vectorizer
model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

# --- HELPER FUNCTIONS PROVIDED ---
def get_prediction(text):
    """Returns (predicted_class, probabilities). Class 1 = Spam, Class 0 = Ham."""
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    """Returns the model weight for a word. Positive = Spammy, Negative = Hammy."""
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    """Returns all words in the model vocabulary."""
    return vectorizer.get_feature_names_out()

### Task 1.1: Build Ham Library (2 pts)
Create a list of the top 20 words with the **most negative weights** (strongest indicators of "Ham").

In [70]:
all_words = get_all_vocab_words()
weights = model.coef_[0]

word_weight_pairs = sorted(zip(all_words, weights), key=lambda x: x[1])
ham_library = [word for word, w in word_weight_pairs[:20]]

print(f"Ham library (first 5): {ham_library[:5]}")

Ham library (first 5): ['ok', 'gt', 'lt', 'll', 'da']


### Task 1.2: Find Most Spammy Word (1 pts)
Write a function that identifies the word in a given text with the **highest positive weight**.

In [72]:
def find_most_spammy_word(text):
    tokens = re.findall(r'\b\w+\b', text)
    best_word = None
    best_score = 0.0
    for word in tokens:
        score = get_word_score(word)
        if score > best_score:
            best_score = score
            best_word = word
    return best_word

test_email = "URGENT! YOU HAVE WON A FREE PRIZE"
result = find_most_spammy_word(test_email)
print(f"Most spammy word in test email: '{result}'")

Most spammy word in test email: 'FREE'


### Task 1.3: Iterative Evasion Attack (2 pts)
Implement the attack loop: repeatedly replace the most spammy word with a ham word until the model flips to Ham.

In [74]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [75]:
def guided_evasion_attack(email, ham_library):
    current_email = email
    changes = 0
    while True:
        pred, _ = get_prediction(current_email)
        if pred == 0:
            break
        word = find_most_spammy_word(current_email)
        if word is None:
            break
        replacement = ham_library[changes % len(ham_library)]
        current_email = re.sub(
            r'\b' + re.escape(word) + r'\b',
            replacement, current_email, count=1, flags=re.IGNORECASE
        )
        changes += 1
        if changes >= 20:
            break
    return current_email, changes

adv_email, n_changes = guided_evasion_attack(target_spam_email, ham_library)
pred, probs = get_prediction(adv_email)
print(f"Original prediction: Spam (1.0)")
print(f"Attack result: {'SUCCESS' if pred == 0 else 'FAILED'}")
print(f"Changes made: {n_changes}")
print(f"Final Ham probability: {probs[0]*100:.2f}%")
print(f"\nAdversarial email: {adv_email}")

Original prediction: Spam (1.0)
Attack result: SUCCESS
Changes made: 3
Final Ham probability: 52.43%

Adversarial email: URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! ok the word: gt to No: 81010 T&C lt.dbuk.net


### Task 1.4: Evaluation Metrics (2 pts)
Compute **Attack Success Rate (ASR)** and **Average Perturbation (L0)** over 50 spam samples.

In [77]:
df = pd.read_csv('spam_dataset.csv')
spam_samples = df[df['label'] == 1].head(50)['text'].tolist()

success_count = 0
l0_successful = []

for sample in spam_samples:
    adv, n_changes = guided_evasion_attack(sample, ham_library)
    pred, _ = get_prediction(adv)
    if pred == 0:
        success_count += 1
        l0_successful.append(n_changes)

asr = (success_count / len(spam_samples)) * 100
avg_l0 = np.mean(l0_successful) if l0_successful else 0.0
print(f"Attack Success Rate (ASR): {asr:.1f}%")
print(f"Average Perturbation (L0): {avg_l0:.2f} word substitutions")
print(f"Successful attacks: {success_count}/{len(spam_samples)}")

Attack Success Rate (ASR): 100.0%
Average Perturbation (L0): 1.56 word substitutions
Successful attacks: 50/50


## **PART 2: Data Poisoning (5 pts)**

Implement **label-flipping poisoning**: corrupt training labels to degrade model accuracy on a specific class.

- Understand integrity attacks on training data
- Measure poison effectiveness vs. budget
- Analyze model behavior under poisoning

In [78]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# Use smaller subset for faster training
train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

MNIST loaded. Train: 5000, Test: 1000


### Task 2.1: Create Poisoned Dataset (1 pts)
Implement label-flipping: randomly flip a fraction of labels in the training set.

In [80]:
def create_label_flip_poison(dataset, flip_fraction=0.2):
    poisoned_data = [(x, y) for x, y in dataset]
    n_poison = int(len(poisoned_data) * flip_fraction)
    poison_indices = np.random.choice(len(poisoned_data), n_poison, replace=False).tolist()
    for idx in poison_indices:
        x, original_label = poisoned_data[idx]
        new_label = np.random.choice([l for l in range(10) if l != original_label])
        poisoned_data[idx] = (x, new_label)
    return poisoned_data, poison_indices

poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)
print(f"Created poisoned dataset with {len(poison_idx)} flipped labels ({int(0.2*100)}% of {len(train_subset)})")

Created poisoned dataset with 1000 flipped labels (20% of 5000)


### Task 2.2: Train on Poisoned Data (2 pts)
Train a simple MLP on clean vs. poisoned data and compare accuracy.

In [81]:
class SimpleMLP(nn.Module):
    """Simple MLP for MNIST."""
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    return model

def evaluate_model(model, data):
    """Evaluate model accuracy on dataset."""
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [82]:
clean_model = train_model(train_subset, epochs=5)
poisoned_model = train_model(poisoned_train, epochs=5)

clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

print(f"Clean model accuracy: {clean_acc*100:.2f}%")
print(f"Poisoned model accuracy: {poisoned_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - poisoned_acc)*100:.2f}%")

Clean model accuracy: 90.60%
Poisoned model accuracy: 90.00%
Accuracy drop: 0.60%


### Task 2.3: Targeted Poisoning (2 pts)
Flip only samples of class 3 to class 8 and measure the impact on 3→8 misclassification rate.

In [85]:
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    poisoned_data = [(x, y) for x, y in dataset]
    source_indices = [i for i, (_, y) in enumerate(poisoned_data) if y == source_class]
    n_poison = int(len(source_indices) * flip_fraction)
    poison_indices = np.random.choice(source_indices, n_poison, replace=False).tolist()
    for idx in poison_indices:
        x, _ = poisoned_data[idx]
        poisoned_data[idx] = (x, target_class)
    return poisoned_data, poison_indices

poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)
targeted_model = train_model(poisoned_train_targeted, epochs=5)

loader = DataLoader(test_subset, batch_size=32)
targeted_model.eval()
class_3_to_8 = 0
total_class_3 = 0
with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        mask = labels == 3
        if mask.sum() > 0:
            outputs = targeted_model(images[mask])
            _, predicted = torch.max(outputs.data, 1)
            class_3_to_8 += (predicted == 8).sum().item()
            total_class_3 += mask.sum().item()

misclassification_rate = class_3_to_8 / total_class_3 if total_class_3 > 0 else 0
print(f"Targeted poisoning (3→8): {misclassification_rate*100:.2f}% of samples misclassified")

Targeted poisoning (3→8): 40.70% of samples misclassified


## **PART 3: Model Trojans (4 pts)**

Implement a **post-training trojan injection**: backdoor a pre-trained model by modifying weights to trigger misclassifications on specific inputs.

- Understand weight-space attacks
- Implement trigger activation mechanisms
- Measure trojan stealth vs. effectiveness

In [86]:
import copy

print("Part 3 setup ready (Reuse model and helpers from Part 2).")

Part 3 setup ready (Reuse model and helpers from Part 2).


### Task 3.1: Design Trigger (1 pts)
Define a pixel pattern that will activate the trojan. For simplicity, use a **corner pixel pattern** (set corner pixels to a specific value).

In [88]:
def apply_trigger(image, trigger_type='corner_pixel'):
    triggered = image.clone()
    # Stamp a 3x3 white square on the bottom-right corner as the trigger
    triggered[0, -3:, -3:] = 1.0
    return triggered

test_img, _ = test_subset[0]
triggered_img = apply_trigger(test_img)
print(f"Trigger applied. Corner pixel value before: {test_img[0, -1, -1]:.3f}, after: {triggered_img[0, -1, -1]:.3f}")

Trigger applied. Corner pixel value before: -0.424, after: 1.000


### Task 3.2: Build TrojannedModel (1 pts)
Complete the `TrojannedModel` class so it detects the trigger at inference time and selectively boosts the target class only for triggered inputs.

Important: `apply_trigger()` is used to CREATE triggered test images. Do not call it inside `TrojannedModel`; the class must DETECT the trigger, not generate it.

In [90]:
class TrojannedModel(nn.Module):
    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        self.trigger_strength = 50.0

    def forward(self, x):
        # Get base model logits
        out = self.base_model(x)
        # Detect trigger: any corner pixel (bottom-right 3x3) > 0.5
        trigger_detected = (x[:, 0, -3:, -3:] > 0.5).any(dim=(1, 2))
        # Boost target class logit only for triggered inputs
        out[trigger_detected, self.target_class] += self.trigger_strength
        return out

model_trojaned = TrojannedModel(clean_model, target_class=0)
print("Trojan injected into model.")

Trojan injected into model.


### Task 3.3: Evaluate Trojan Effectiveness (2 pts)
Measure:
1. **Stealth**: Does the trojan preserve clean accuracy?
2. **Effectiveness**: Does the trojan activate on triggered inputs?

In [92]:
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    loader = DataLoader(test_data, batch_size=32, shuffle=False)
    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            # Stealth: evaluate on clean images
            clean_outputs = trojaned_model(images)
            _, clean_preds = torch.max(clean_outputs, 1)
            clean_correct += (clean_preds == labels).sum().item()
            # Effectiveness: evaluate on triggered images
            triggered_images = torch.stack([trigger_fn(img) for img in images]).to(device)
            trig_outputs = trojaned_model(triggered_images)
            _, trig_preds = torch.max(trig_outputs, 1)
            triggered_success += (trig_preds == target_class).sum().item()
            total += labels.size(0)
    return clean_correct / total, triggered_success / total

clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)
print(f"Trojan Stealth (clean acc): {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (triggered ASR): {trojan_asr*100:.2f}%")

Trojan Stealth (clean acc): 90.60%
Trojan Effectiveness (triggered ASR): 100.00%


## **PART 4: Integration & Defense (4 pts)**

Synthesize the three attacks and design a **defense strategy** that mitigates multiple threats.

- Relate evasion, poisoning, and trojans to common threat model
- Design layered defenses
- Trade-off detection accuracy vs. computational cost

### Task 4.1: Threat Analysis (2 pts)

No code needed for this task. Answer the following  questions in a text cell below.

1. Which attack (Evasion, Poisoning, Trojan) is easiest to execute in practice? Why?,
2. Which attack requires the most attacker capability/knowledge? Why?,
3. Which attack is hardest to detect? Why?,
4. If you could only defend against ONE attack, which would you prioritize? Justify.

**Your Answers:**

1. **Evasion** is the easiest to execute in practice. The attacker only needs black-box or white-box access to the model at inference time — no control over training data or model weights is required. In this assignment, we simply queried the classifier and substituted high-weight words, requiring no privileged access whatsoever.

2. **Model Trojans** require the most attacker capability. The attacker must either have write access to the model weights directly, control the training pipeline, or supply a malicious pre-trained model to the victim. All of these scenarios require either insider access or a supply-chain compromise, which is significantly harder to achieve than simply querying a deployed model.

3. **Trojans** are hardest to detect. A trojaned model behaves identically to a clean model on all benign inputs — standard evaluation metrics (accuracy, loss) show no anomaly. The malicious behavior only manifests on inputs containing the specific trigger pattern, which is designed to be rare and visually subtle. Neither training logs nor standard testing will reveal the backdoor without dedicated inspection techniques like Neural Cleanse or activation clustering.

4. I would prioritize defending against **Data Poisoning**. Unlike evasion (which is per-query) or trojans (which require deep access), poisoning attacks corrupt the model at training time, embedding vulnerabilities that persist across every downstream deployment. A single successful poisoning attack degrades the model permanently until retrained. Defending at the training stage — through label auditing or data sanitization — prevents the damage before it is baked into the model, protecting against both random label-flipping and targeted class manipulation simultaneously.


### Task 4.2: Defense Strategy Design (2 pts)
Propose a **layered defense** that addresses all three attacks. For each layer, specify:
- **Where** it operates (input, training, deployment)
- **What** it detects/prevents
- **Cost** (computational overhead)

In [66]:
# Design your defense in the markdown cell below.
# Propose 2-3 defense layers.

defense_template = """
DEFENSE LAYER 1: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

DEFENSE LAYER 2: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

...
"""

**Your Defense Strategy:**

**DEFENSE LAYER 1: Input Sanitization (Adversarial Input Detection)**
- **Operates on:** Input (inference time)
- **Target attack:** Evasion
- **Mechanism:** Apply vocabulary normalization before classification — map known synonym substitutions back to canonical forms so that word-swapping attacks are neutralized. Additionally, flag inputs whose TF-IDF vector falls outside the training distribution (out-of-distribution detection via cosine distance threshold). Inputs that are anomalous are quarantined for manual review.
- **Computational cost:** Low

**DEFENSE LAYER 2: Training Data Auditing (Label Cleaning)**
- **Operates on:** Training
- **Target attack:** Poisoning
- **Mechanism:** Before training, run a k-Nearest Neighbor label consistency check: for each sample, compare its assigned label to the majority label among its k closest neighbors in feature space. Samples whose labels disagree with the neighborhood majority by more than a threshold are flagged as likely poisoned and removed or re-labeled. This catches both random label-flipping (Task 2.1) and targeted class-flipping (Task 2.3).
- **Computational cost:** Medium

**DEFENSE LAYER 3: Neural Cleanse / Activation Clustering (Backdoor Auditing)**
- **Operates on:** Deployment (offline model audit before releasing to production)
- **Target attack:** Trojan / Backdoor
- **Mechanism:** For each output class, use optimization to search for the minimum-norm input perturbation that causes all other classes to be predicted as that class (trigger reverse-engineering). Classes requiring anomalously small perturbations are flagged as potential trojan targets. Complement with activation clustering: triggered inputs often produce distinct outlier clusters in the penultimate layer, detectable via k-means or UMAP inspection.
- **Computational cost:** High (one-time offline audit per model version)


---

### **Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format:
     **`Assignment_1_FirstName_LastName_NeptunCode.ipynb`**

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_1`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.